In [ ]:
# NOTE: If Colab updates its PyTorch or CUDA version, these wheel URLs
# will need updating. Check:
# https://github.com/state-spaces/mamba/releases
# https://github.com/Dao-AILab/causal-conv1d/releases
# Match: cu{CUDA_MAJOR}, torch{TORCH_VERSION}, cp{PYTHON_VERSION}
import subprocess, sys, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")

# Core dependencies (fast)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "miditok>=3.0.0", "ncps>=0.0.7", "safetensors>=0.4.0",
    "pretty_midi>=0.2.10", "einops>=0.7.0"], check=False)

# Pre-built wheels for mamba - no compilation
# These wheels match: Python 3.12, PyTorch 2.10, CUDA 12.x (Colab current)
CAUSAL_CONV1D_WHEEL = (
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1/"
    "causal_conv1d-1.6.1+cu128torch2.10cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
)
MAMBA_WHEEL = (
    "https://github.com/state-spaces/mamba/releases/download/v2.3.1/"
    "mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)

cuda_ver = torch.version.cuda or ""
py_ver = f"{sys.version_info.major}{sys.version_info.minor}"

if torch.cuda.is_available():
    print("GPU detected - installing pre-built Mamba wheels...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        CAUSAL_CONV1D_WHEEL], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        MAMBA_WHEEL], capture_output=True)
    if r2.returncode != 0:
        print("Mamba wheel install failed - GRU fallback will be used")
        print(r2.stderr.decode()[-500:])
    else:
        print("Mamba installed successfully from pre-built wheel")
else:
    print("No GPU - GRU fallback will be used for local development")

# Verify
try:
    import mamba_ssm
    print(f"mamba-ssm {mamba_ssm.__version__} active - real Mamba enabled")
except ImportError:
    print("mamba-ssm not available - GRU fallback active")

sys.path.insert(0, '/content/piano_midi_model')
print("Setup complete.")


In [ ]:
# ============================================
# EDIT THIS CELL TO CHANGE TRAINING SCALE
# ============================================
SCALE = "nano"       # Options: nano, micro, small, medium
MAX_EPOCHS = 200     # Total target epochs (across all sessions)
# ============================================

from scale_config import list_presets, get_preset
list_presets()
print(f"\nSelected: {SCALE}")
get_preset(SCALE)


In [ ]:
# Optional: parameter calibration on current runtime (no training)
from session import calibrate_preset
for preset_name in ['nano', 'micro', 'small', 'medium']:
    calibrate_preset(preset_name)


In [ ]:
from session import run_session
run_session(scale=SCALE)


In [ ]:
# Generate a continuation from a specific seed file
from generation.generate import generate_continuation, GenerationConfig
from drive_sync import DriveSync

sync = DriveSync()
# List available seed files
import os
from pathlib import Path
seed_files = list(Path("path/to/seeds").glob("*.mid"))
print(f"Found {len(seed_files)} seed files")

gen_config = GenerationConfig(
    max_new_tokens=1024,
    temperature=0.9,
    top_p=0.95,
)
# generate_continuation(model, tokenizer, seed_files[0], "output.mid", ...)


In [ ]:
from drive_sync import DriveSync
sync = DriveSync()
sync.mount()
history = sync.get_training_history()
# Plot loss curves across all sessions
import matplotlib.pyplot as plt
epochs = [s['end_epoch'] for s in history['sessions']]
val_losses = [s['final_val_loss'] for s in history['sessions']]
plt.figure(figsize=(12, 4))
plt.plot(epochs, val_losses, 'o-', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Val Loss')
plt.title('Training progress across all sessions')
plt.grid(True, alpha=0.3)
plt.show()
